# 02 — QASPER: chunking + in-paper retrieval

Steps:
1. Load QASPER (test split) and look at one paper and one question.
2. Turn papers into paragraphs, map gold evidence to paragraphs.
3. Fixed evaluation subset (300 questions), saved to `results/qasper_eval_subset.json`.
4. Three token-based chunking strategies (sizes measured with the E5 tokenizer, so no chunk is truncated by the models).
5. BM25 / Dense / Hybrid / Hybrid + Cross-Encoder **inside each paper**, reusing `src/retrieval.py`.
6. Results table: method × chunking.
7. Paired bootstrap confidence intervals for the key comparisons.
8. Save rankings for the generation stage.
9. Error analysis.

Colab: Runtime → Change runtime type → T4 GPU (needed for sections 5–6 only).

In [ ]:
# Colab only: mount Drive and go to the repository folder
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/<path-to>/hybrid-multilingual-rag-retrieval"
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import sys, json, pandas as pd

cwd = Path.cwd()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(ROOT))

from src.retrieval import load_dense_model
from src.qasper import download_qasper, load_qasper_split, paper_paragraphs, extract_questions, sample_eval_subset
from src.chunking import make_chunkers, gold_chunk_ids
from src.qasper_retrieval import PaperIndex, embed_all_chunks, evaluate_qasper, paired_bootstrap

## 1. Load QASPER

In [ ]:
DATA = ROOT / "datasets" / "qasper"
download_qasper(DATA)
papers = load_qasper_split("test", DATA)
print("papers:", len(papers), "| questions:", sum(len(p["qas"]) for p in papers.values()))

### One paper, one question

In [ ]:
pid = next(iter(papers))
paper = papers[pid]
print(paper["title"], "\n")
print("sections:", [s["section_name"] for s in paper["full_text"]][:15])
qa = paper["qas"][0]
print("\nQUESTION:", qa["question"])
for ann in qa["answers"]:
    a = ann["answer"]
    print("\n-- annotator answer:", a["extractive_spans"] or a["free_form_answer"] or a["yes_no"] or "UNANSWERABLE")
    for ev in a["evidence"]:
        print("   evidence:", ev[:200])

## 2. Paragraphs + gold evidence

In [ ]:
records, ev_stats = extract_questions(papers)
print(ev_stats)
print("questions with text evidence:", sum(bool(r["gold_paragraphs"]) for r in records), "/", len(records))
pd.Series([r["answers"][0]["type"] for r in records]).value_counts()

## 3. Fixed evaluation subset (reused by ALL later experiments)

In [ ]:
eval_qs = sample_eval_subset(records, n=300, seed=42, path=ROOT / "results" / "qasper_eval_subset.json")
eval_papers = sorted({r["paper_id"] for r in eval_qs})
print(len(eval_qs), "questions from", len(eval_papers), "papers")
paragraphs_by_paper = {pid: paper_paragraphs(papers[pid]) for pid in eval_papers}

## 4. Token-based chunking

Sizes are counted with the tokenizer of `multilingual-e5-base`:
- **paragraph**: one paragraph = one chunk; paragraphs longer than 256 tokens are split inside the paragraph (all pieces keep the same paragraph id);
- **paragraph+section**: same, with the section name prepended (counted in the budget);
- **fixed256**: 256-token windows with 64-token overlap, ignoring paragraph boundaries.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
MAX_TOKENS = 256
CHUNKERS = make_chunkers(tokenizer, MAX_TOKENS)

chunks = {}   # strategy -> {paper_id: [chunks]}
for name, chunker in CHUNKERS.items():
    chunks[name] = {pid: chunker(pid, paragraphs_by_paper[pid]) for pid in eval_papers}

# how many original paragraphs were too long and had to be split
para_tokens = [len(tokenizer(p["text"], add_special_tokens=False)["input_ids"])
               for ps in paragraphs_by_paper.values() for p in ps]
n_long = sum(t > MAX_TOKENS for t in para_tokens)
print(f"paragraphs longer than {MAX_TOKENS} tokens: {n_long} / {len(para_tokens)} ({100*n_long/len(para_tokens):.1f}%)")
print("longest paragraph:", max(para_tokens), "tokens")

rows = []
for name, by_paper in chunks.items():
    all_c = [c for cs in by_paper.values() for c in cs]
    toks = [c["n_tokens"] for c in all_c]
    rows.append({"chunking": name, "chunks/paper": round(len(all_c) / len(by_paper), 1),
                 "mean tokens": round(sum(toks) / len(toks), 1), "max tokens": max(toks)})
pd.DataFrame(rows)

In [ ]:
# gold paragraphs that were long enough to be split
long_gold = sum(1 for r in eval_qs for i in r["gold_paragraphs"]
                if len(tokenizer(paragraphs_by_paper[r["paper_id"]][i]["text"],
                                                  add_special_tokens=False)["input_ids"]) > MAX_TOKENS)
total_gold = sum(len(r["gold_paragraphs"]) for r in eval_qs)
print(f"gold paragraphs split into several chunks: {long_gold} / {total_gold}")

In [ ]:
# sanity check: one chunk of each type
for name in chunks:
    c = chunks[name][eval_papers[0]][3]
    print(f"[{name}] {c['chunk_id']} paras={c['para_ids']} tokens={c['n_tokens']}\n{c['text'][:300]}\n")

## 5. Models + indexes

In [ ]:
from sentence_transformers import CrossEncoder

dense_model = load_dense_model("intfloat/multilingual-e5-base")
dense_model.max_seq_length = 512                                    # explicit: chunks are <= 256 tokens
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)  # question + chunk fit in 512

indexes, chunk_info, gold = {}, {}, {}
for name, by_paper in chunks.items():
    emb = embed_all_chunks(dense_model, by_paper, cache_path=ROOT / "cache" / f"qasper_e5_{name}_{MAX_TOKENS}tok.npz")
    indexes[name] = {pid: PaperIndex(by_paper[pid], emb[pid]) for pid in eval_papers}
    chunk_info[name] = {c["chunk_id"]: c for cs in by_paper.values() for c in cs}
    gold[name] = {r["question_id"]: gold_chunk_ids(by_paper[r["paper_id"]], r["gold_paragraphs"]) for r in eval_qs}

## 6. Evaluation: method × chunking

In [ ]:
def make_search_fns(name):
    idx = indexes[name]
    return {
        "BM25":   lambda r, k: idx[r["paper_id"]].bm25_search(r["question"], k),
        "Dense":  lambda r, k: idx[r["paper_id"]].dense_search(r["question"], dense_model, k),
        "Hybrid": lambda r, k: idx[r["paper_id"]].hybrid_search(r["question"], dense_model, k),
        "Hybrid+CE": lambda r, k: idx[r["paper_id"]].reranked_search(r["question"], dense_model, reranker, k),
    }

results, all_rankings, per_question = [], {}, {}
for name in chunks:
    for method, fn in make_search_fns(name).items():
        m, rankings, per_q = evaluate_qasper(fn, eval_qs, gold[name], chunk_info[name])
        results.append({"chunking": name, "method": method, **m})
        all_rankings[(name, method)] = rankings
        per_question[(name, method)] = per_q
        print(name, method, "done")

df = pd.DataFrame(results).round(3)
df.to_csv(ROOT / "results" / "qasper_retrieval_metrics.csv", index=False)
df

In [ ]:
cols = ["chunking", "method", "ParaRecall@1", "ParaRecall@3", "ParaRecall@5", "MRR@5", "Tokens@5"]
df[cols]

## 7. Paired bootstrap (95% CI)

Difference B − A in ParaRecall@5 on the same 300 questions. If the interval contains 0, the difference is within noise.

In [ ]:
P, PS, F = "paragraph", "paragraph+section", f"fixed{MAX_TOKENS}"
comparisons = [
    ((P, "BM25"),   (P, "Dense")),
    ((P, "Dense"),  (P, "Hybrid")),
    ((P, "Hybrid"), (P, "Hybrid+CE")),
    ((P, "Dense"),  (P, "Hybrid+CE")),
    ((P, "BM25"),   (PS, "BM25")),
    ((P, "Hybrid+CE"), (PS, "Hybrid+CE")),
    ((PS, "Hybrid+CE"), (F, "Hybrid+CE")),
]
metric = "ParaRecall@5"
rows = []
for a, b in comparisons:
    d, lo, hi = paired_bootstrap(per_question[a][metric], per_question[b][metric])
    rows.append({"A": " / ".join(a), "B": " / ".join(b), f"Δ {metric}": round(d, 3),
                 "95% CI": f"[{lo:.3f}, {hi:.3f}]", "significant": not (lo <= 0 <= hi)})
boot = pd.DataFrame(rows)
boot.to_csv(ROOT / "results" / "qasper_bootstrap.csv", index=False)
boot

## 8. Save rankings for the generation stage

In [ ]:
out = ROOT / "results" / "qasper_rankings.jsonl"
with open(out, "w", encoding="utf-8") as f:
    for (name, method), rankings in all_rankings.items():
        for qid, ranked in rankings.items():
            f.write(json.dumps({"chunking": name, "method": method, "question_id": qid, "ranked": ranked}) + "\n")
print("saved", out)

## 9. Error analysis

Questions where one setup finds gold evidence in the top 5 and another does not.
Uses the rankings saved in section 8, so no models or GPU are needed (run sections 1–4 first).
For each pattern: number of questions and up to 3 examples (question, gold paragraph, top-1 chunk of each setup).

In [ ]:
from collections import defaultdict

R = defaultdict(dict)
with open(ROOT / "results" / "qasper_rankings.jsonl", encoding="utf-8") as f:
    for line in f:
        x = json.loads(line)
        R[(x["chunking"], x["method"])][x["question_id"]] = x["ranked"]

gold_sets = {name: {r["question_id"]: set(gold_chunk_ids(chunks[name][r["paper_id"]], r["gold_paragraphs"]))
                    for r in eval_qs} for name in chunks}
info = {name: {c["chunk_id"]: c for cs in chunks[name].values() for c in cs} for name in chunks}
qs = {r["question_id"]: r for r in eval_qs}

def gold_rank(name, method, qid, k=5):
    for i, c in enumerate(R[(name, method)][qid][:k], 1):
        if c in gold_sets[name][qid]:
            return i
    return None

def find(fail, success, k=5):
    """fail / success = (chunking, method)"""
    return [q for q in qs if gold_rank(*fail, q, k) is None and gold_rank(*success, q, k) is not None]

def show(qid, setups):
    r = qs[qid]
    print("QUESTION:", r["question"], f"  [{qid}]")
    name0 = setups[0][0]
    g = sorted(gold_sets[name0][qid])[0]
    print("GOLD:", info[name0][g]["text"][:300], "...")
    for name, m in setups:
        top1 = info[name][R[(name, m)][qid][0]]["text"][:150]
        print(f"  {name} / {m}: gold at rank {gold_rank(name, m, qid)} | top-1: {top1}...")
    print()

P, PS = "paragraph", "paragraph+section"
cases = {
    "BM25 fails, Dense succeeds": ((P, "BM25"), (P, "Dense")),
    "Dense succeeds, Hybrid fails (RRF hurts)": ((P, "Hybrid"), (P, "Dense")),
    "Hybrid fails, Hybrid+CE succeeds": ((P, "Hybrid"), (P, "Hybrid+CE")),
    "Section name helps BM25": ((P, "BM25"), (PS, "BM25")),
}
for title, (fail, success) in cases.items():
    ids = find(fail, success)
    print(f"===== {title}: {len(ids)} questions =====\n")
    for qid in ids[:3]:
        show(qid, [fail, success])